In [ ]:
pip install transformers

In [ ]:
# import json
# from pathlib import Path

# from transformers import AutoTokenizer


# def load_json(path: Path):
#     with path.open("r", encoding="utf-8") as f:
#         return json.load(f)


# def build_pair_input(claim_text: str, evidence_ids: list[str], evidence_db: dict[str, str]) -> tuple[str, str]:
#     evidence_texts = [evidence_db[eid] for eid in evidence_ids if eid in evidence_db]
#     evidence_joined = " [SEP] ".join(evidence_texts)
#     return claim_text, evidence_joined


# def check_claim_file(file_path: Path, tokenizer, evidence_db: dict[str, str], model_max_len: int, list_limit: int):
#     claims = load_json(file_path)
#     overflow_cases = []
#     total = len(claims)

#     sample_value = next(iter(claims.values())) if claims else {}
#     if "evidences" not in sample_value:
#         print(f"\n=== {file_path.name} ===")
#         print(f"total claims: {total}")
#         print("skip: this file has no gold 'evidences' field, cannot compute overflow with full evidence.")
#         return

#     for claim_id, item in claims.items():
#         claim_text = item["claim_text"]
#         evidence_ids = item.get("evidences", [])
#         text_a, text_b = build_pair_input(claim_text, evidence_ids, evidence_db)

#         tokenized = tokenizer(
#             text_a,
#             text_b,
#             add_special_tokens=True,
#             truncation=False,
#             return_attention_mask=False,
#             return_token_type_ids=False,
#         )
#         token_len = len(tokenized["input_ids"])
#         overflow = max(0, token_len - model_max_len)
#         if overflow > 0:
#             overflow_cases.append(
#                 {
#                     "claim_id": claim_id,
#                     "token_len": token_len,
#                     "overflow": overflow,
#                     "n_evidences": len(evidence_ids),
#                 }
#             )

#     overflow_cases.sort(key=lambda x: x["overflow"], reverse=True)

#     print(f"\n=== {file_path.name} ===")
#     print(f"total claims: {total}")
#     print(f"max input tokens: {model_max_len}")
#     print(f"overflow claims: {len(overflow_cases)}")

#     if overflow_cases:
#         print(f"top {min(list_limit, len(overflow_cases))} overflow cases:")
#         for i, row in enumerate(overflow_cases[:list_limit], 1):
#             print(
#                 f"{i:02d}. {row['claim_id']} | tokens={row['token_len']} | "
#                 f"overflow=+{row['overflow']} | evidences={row['n_evidences']}"
#             )


# def main():
#     # Removed argparse and directly define parameters for Colab execution
#     data_dir_str = "data"
#     model_name = "bert-base-uncased"
#     max_len = 512
#     list_limit = 10

#     data_dir = Path(data_dir_str)
#     evidence_path = data_dir / "evidence.json"
#     train_path = data_dir / "train-claims.json"
#     test_path = data_dir / "test-claims-unlabelled.json"

#     evidence_db = load_json(evidence_path)
#     tokenizer = AutoTokenizer.from_pretrained(model_name)

#     check_claim_file(train_path, tokenizer, evidence_db, max_len, list_limit)
#     check_claim_file(test_path, tokenizer, evidence_db, max_len, list_limit)


# if __name__ == "__main__":
#     main()

In [ ]:
# =========================
# Cell 1: Imports
# =========================
import json
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

In [ ]:
# =========================
# Cell 2: Config
# =========================
@dataclass
class Config:
    data_dir: str = "data"
    model_name: str = "bert-base-uncased"
    output_dir: str = "outputs/bert_claim_classifier"
    max_length: int = 512
    seed: int = 42
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    train_batch_size: int = 8
    eval_batch_size: int = 8
    num_train_epochs: int = 5
    warmup_ratio: float = 0.1
    evid_sep: str = "[EVID_SEP]"


config = Config()

label2id = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2,
    "DISPUTED": 3,
}
id2label = {v: k for k, v in label2id.items()}

In [ ]:
# =========================
# Cell 3: Reproducibility
# =========================
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(config.seed)

In [ ]:
# =========================
# Cell 4: Data Loading
# =========================
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


data_dir = Path(config.data_dir)
train_claims = load_json(data_dir / "train-claims.json")
dev_claims = load_json(data_dir / "dev-claims.json")
evidence_db = load_json(data_dir / "evidence.json")

print(f"Train claims: {len(train_claims)}")
print(f"Dev claims:   {len(dev_claims)}")
print(f"Evidence:     {len(evidence_db)}")

Train claims: 1228
Dev claims:   154
Evidence:     1208827


In [ ]:
# =========================
# Cell 5: Build Text Pairs
# =========================
def build_text_b(evidence_ids, evidence_db, evid_sep: str) -> str:
    evidence_texts = [evidence_db[eid] for eid in evidence_ids if eid in evidence_db]
    return f" {evid_sep} ".join(evidence_texts)


def build_examples(claim_dict, evidence_db, evid_sep: str):
    examples = []
    for claim_id, item in claim_dict.items():
        claim_text = item["claim_text"]
        text_b = build_text_b(item.get("evidences", []), evidence_db, evid_sep)
        label_str = item["claim_label"]
        examples.append(
            {
                "claim_id": claim_id,
                "text_a": claim_text,
                "text_b": text_b,
                "label": label2id[label_str],
            }
        )
    return examples


train_examples = build_examples(train_claims, evidence_db, config.evid_sep)
dev_examples = build_examples(dev_claims, evidence_db, config.evid_sep)

print("Sample example keys:", train_examples[0].keys())

Sample example keys: dict_keys(['claim_id', 'text_a', 'text_b', 'label'])


In [ ]:
# =========================
# Cell 6: Tokenizer + Dataset
# =========================
tokenizer = AutoTokenizer.from_pretrained(config.model_name)


def tokenize_batch(batch):
    return tokenizer(
        batch["text_a"],
        batch["text_b"],
        max_length=config.max_length,
        truncation="only_second",
    )


train_ds = Dataset.from_list(train_examples)
dev_ds = Dataset.from_list(dev_examples)

train_ds = train_ds.map(tokenize_batch, batched=True)
dev_ds = dev_ds.map(tokenize_batch, batched=True)

train_ds = train_ds.remove_columns(["claim_id", "text_a", "text_b"])
dev_ds = dev_ds.remove_columns(["claim_id", "text_a", "text_b"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/1228 [00:00<?, ? examples/s]

Map:   0%|          | 0/154 [00:00<?, ? examples/s]

In [ ]:
# =========================
# Cell 7: Model + Metrics
# =========================
model = AutoModelForSequenceClassification.from_pretrained(
    config.model_name,
    num_labels=4,
    label2id=label2id,
    id2label=id2label,
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# =========================
# Cell 8: Training Setup
# =========================
training_args = TrainingArguments(
    output_dir=config.output_dir,
    seed=config.seed,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
    per_device_train_batch_size=config.train_batch_size,
    per_device_eval_batch_size=config.eval_batch_size,
    num_train_epochs=config.num_train_epochs,
    warmup_ratio=config.warmup_ratio,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
# =========================
# Cell 9: Train + Evaluate
# =========================
train_result = trainer.train()
print("Train result:", train_result.metrics)

dev_metrics = trainer.evaluate()
print("Dev metrics:", dev_metrics)

Epoch,Training Loss,Validation Loss,Accuracy
1,0.992420,0.954152,0.603896
2,0.841239,0.941637,0.655844
3,0.676518,0.985042,0.668831
4,0.500662,0.927364,0.727273
5,0.457740,0.810745,0.727273


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Train result: {'train_runtime': 634.8903, 'train_samples_per_second': 9.671, 'train_steps_per_second': 1.213, 'total_flos': 877935134499072.0, 'train_loss': 0.7511113563141265, 'epoch': 5.0}


Dev metrics: {'eval_loss': 0.9268856048583984, 'eval_accuracy': 0.7272727272727273, 'eval_runtime': 2.8676, 'eval_samples_per_second': 53.704, 'eval_steps_per_second': 6.975, 'epoch': 5.0}
